In [3]:
import os
import re
import glob
from datetime import date, datetime, timedelta
from pathlib import Path
from decimal import Decimal, InvalidOperation
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load environment variables
load_dotenv()

# ==================== CONFIG ====================
# Database Configuration
DB_CONFIG = {
    "host": os.getenv("DB_HOST"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "database": os.getenv("DB_NAME"),
    "port": os.getenv("DB_PORT", 3306),
}

# Paths


# Database
TABLE_NAME = "warranty_history"

# Column Mapping
COL_MAP = {
    "ID": "id",
    "Ngày tạo": "created_at",
    "Kho hàng": "warehouse",
    "Khách hàng": "customer_name",
    "Số điện thoại": "customer_phone",
    "Địa chỉ": "customer_address",
    "Ngày mua": "purchase_date",
    "Sản phẩm": "product_name",
    "Mã sản phẩm": "product_code",
    "Trạng thái sản phẩm": "product_status",
    "Trạng thái phiếu bảo hành": "warranty_ticket_status",
    "IMEI": "imei",
    "Loại": "warranty_type",
    "Trung tâm bảo hành": "service_center",
    "Ngày hẹn lấy TTBH": "pickup_date_service_center",
    "Chi phí sửa chữa": "repair_cost",
    "Phí sửa chữa báo khách": "repair_fee_customer",
    "Lý do": "reason",
    "Trạng thái": "status",
    "Ngày hẹn trả": "return_due_date",
    "Ngày trả khách": "return_date",
    "Hình thức trả khách": "return_method",
    "Trả cho khách": "return_to_customer",
    "Người tiếp nhận": "received_by",
    "Người sửa": "repaired_by",
    "Linh kiện": "spare_part",
    "SL linh kiện": "spare_part_qty",
    "Giá linh kiện": "spare_part_price",
    "Ghi chú": "note",
    "Ghi chú CSKH": "customer_service_note",
}

DATE_COLS = [
    "created_at",
    "purchase_date",
    "pickup_date_service_center",
    "return_due_date",
    "return_date",
]
MONEY_COLS = ["repair_cost", "repair_fee_customer", "spare_part_price"]
INT_COLS = ["spare_part_qty"]


# ==================== DATABASE ====================
def create_db_engine():
    """Create MySQL database engine"""
    connection_string = (
        f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    )
    return create_engine(connection_string)


def create_table_if_not_exists(engine):
    """Create warranty_history table if it doesn't exist"""
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
        id BIGINT,
        created_at DATETIME,
        warehouse VARCHAR(255),
        customer_name VARCHAR(255),
        customer_phone VARCHAR(50),
        customer_address TEXT,
        purchase_date DATETIME,
        product_name VARCHAR(500),
        product_code VARCHAR(100),
        product_status VARCHAR(100),
        warranty_ticket_status VARCHAR(100),
        imei VARCHAR(100),
        warranty_type VARCHAR(100),
        service_center VARCHAR(255),
        pickup_date_service_center DATETIME,
        repair_cost DECIMAL(15,2),
        repair_fee_customer DECIMAL(15,2),
        reason TEXT,
        status VARCHAR(100),
        return_due_date DATETIME,
        return_date DATETIME,
        return_method VARCHAR(100),
        return_to_customer VARCHAR(255),
        received_by VARCHAR(255),
        repaired_by VARCHAR(255),
        spare_part VARCHAR(500),
        spare_part_qty INT,
        spare_part_price DECIMAL(15,2),
        note TEXT,
        customer_service_note TEXT,
        run_from DATETIME,
        run_to DATETIME,
        loaded_at DATETIME,
        PRIMARY KEY (id, created_at),
        INDEX idx_created_at (created_at),
        INDEX idx_customer_phone (customer_phone),
        INDEX idx_imei (imei),
        INDEX idx_loaded_at (loaded_at)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
    """
    
    with engine.begin() as conn:
        conn.execute(text(create_table_sql))
    
    print(f"✅ Table '{TABLE_NAME}' ready")

In [4]:
def db_delete_then_insert(engine, df: pd.DataFrame, run_from: date, run_to: date):
    from_dt = datetime.combine(run_from, datetime.min.time())
    to_dt_plus1 = datetime.combine(run_to + timedelta(days=1), datetime.min.time())

    with engine.begin() as conn:
        # 1) Delete
        result = conn.execute(
            text(f"""
                DELETE FROM {TABLE_NAME}
                WHERE created_at >= :from_dt
                  AND created_at <  :to_dt_plus1
            """),
            {"from_dt": from_dt, "to_dt_plus1": to_dt_plus1},
        )
        print(f"   🗑️  Deleted {result.rowcount} existing rows")

        # 2) Insert (dùng cùng transaction/connection)
        df.to_sql(
            TABLE_NAME,
            con=conn,              # <<< quan trọng
            if_exists="append",
            index=False,
            method="multi",
            chunksize=2000,
        )
        print(f"   💾 Inserted {len(df)} new rows")

In [5]:
# ==================== DATA PROCESSING ====================
def clean_money(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s == "-":
        return None

    s = s.replace("\u00a0", " ").strip()

    # giữ dấu âm
    neg = "-" if s.startswith("-") else ""
    s = s.lstrip("-")

    # remove currency text
    s = re.sub(r"[^\d\.,]", "", s)

    if s == "":
        return None

    # Nếu có cả . và , => assume 1.234,56
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        # Nếu chỉ có , thì thường là phân tách hàng nghìn: 1,234,567
        # (nếu hệ thống bạn có tiền lẻ, đổi logic tại đây)
        s = s.replace(",", "")

    try:
        return float(Decimal(neg + s))
    except (InvalidOperation, ValueError):
        return None

In [6]:
def normalize_df(df: pd.DataFrame, run_from: date, run_to: date) -> pd.DataFrame:
    """Normalize and clean DataFrame"""
    print(f"   📊 Original shape: {df.shape}")
    
    # Rename columns
    df = df.rename(columns=COL_MAP)

    # Keep only mapped columns
    keep_cols = [v for v in COL_MAP.values() if v in df.columns]
    df = df[keep_cols].copy()

    # Convert date columns
    for c in DATE_COLS:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=True)

    # Convert money columns
    for c in MONEY_COLS:
        if c in df.columns:
            df[c] = df[c].apply(clean_money)

    # Convert integer columns
    for c in INT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # Convert ID column
    if "id" in df.columns:
        df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")

    # Add metadata columns
    df["run_from"] = pd.to_datetime(run_from)
    df["run_to"] = pd.to_datetime(run_to)
    df["loaded_at"] = pd.to_datetime(datetime.now())

    # Remove completely empty rows
    df = df.dropna(how="all")
    
    print(f"   📊 Cleaned shape: {df.shape}")
    
    return df

In [7]:
def find_warranty_files(directory: Path = SCRIPT_DIR, pattern: str = "dump_warranty*.xlsx") -> list:
    """Find all warranty Excel files matching pattern"""
    files = list(directory.glob(pattern))
    # Sort by modification time, newest first
    files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return files


def read_excel(path: Path) -> pd.DataFrame:
    """Read Excel file"""
    print(f"   📖 Reading: {path.name}")
    return pd.read_excel(path, engine="openpyxl")

NameError: name 'SCRIPT_DIR' is not defined